# 01 - Feasibility pilot: evidence reliance on public chest X-rays

**What this notebook does**
1. Counts real finding-flip pairs in **CheXTemporal** (labels only, downloaded from Hugging Face).
2. Counts within-patient finding flips in **NIH ChestX-ray14**, for all 14 findings and both directions.
3. Builds *anchor triples* for one finding: anchor image A, a same-patient image F where the finding status is **opposite** (answer should change), and a same-patient image S with the **same** status (answer should not change). Both partners share A's view, so they carry the same kind of nuisance change (another day, positioning, exposure).
4. Scores an open VLM's P(yes) on each image and estimates, with patient-clustered bootstrap CIs:
   - **S** = mean |P(yes|F) - P(yes|A)| (sensitivity)
   - **N** = mean |P(yes|S) - P(yes|A)| (nuisance sensitivity)
   - **R** = S - N (evidence-specific reliance)
   - **direction** = mean (P(yes|F) - P(yes|A)) x (label_F - label_A) (> 0 means it moved the right way)

**Settings before running**
- *Add Input*: dataset **nih-chest-xrays/data** (NIH Chest X-ray Dataset).
- *Accelerator*: **GPU T4 x1** (or P100).  *Internet*: **On** (needed for Hugging Face).
- Optional: for gated models (e.g. MedGemma) accept the licence on Hugging Face and add a Kaggle secret `HF_TOKEN`.

**Caveats:** NIH labels are text-mined from reports (stated accuracy > 90%), so some 'flips' are label noise. This is a pilot to check feasibility and the pipeline, not a result.

In [1]:
# Qwen2.5-VL needs a recent transformers; uncomment if the import in step 6 fails
# !pip -q install -U transformers accelerate

In [2]:
%%writefile pilot_lib.py
# ---- Pair construction and estimation (no model code) ----
import numpy as np
import pandas as pd

NIH_FINDINGS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion", "Emphysema",
                "Fibrosis", "Hernia", "Infiltration", "Mass", "Nodule", "Pleural_Thickening",
                "Pneumonia", "Pneumothorax"]


def add_label(df, finding):
    out = df.copy()
    out["label"] = out["Finding Labels"].str.split("|").apply(lambda L: int(finding in L))
    return out


def build_anchor_triples(df, finding, same_view=True, max_anchors_per_patient=2, seed=0):
    """For each anchor image A, find in the SAME patient:
       flip partner  F: another study with the opposite label for `finding` (should change the answer)
       same partner  S: another study with the same label (should not change the answer)
    Partners are the nearest follow-ups (smallest |follow-up gap|) and share A's view if same_view.
    Using a same-patient, different-day image as the control means both arms carry the same kind
    of nuisance change (positioning, exposure, time), so R = S - N subtracts it."""
    rng = np.random.default_rng(seed)
    d = add_label(df, finding)
    key = ["Patient ID", "View Position"] if same_view else ["Patient ID"]
    d = d[d.groupby(key)["Image Index"].transform("size") >= 3]
    rows = []
    for k, g in d.groupby(key):
        pid = k[0] if isinstance(k, tuple) else k
        img = g["Image Index"].to_numpy(); lab = g["label"].to_numpy()
        fu = g["Follow-up #"].to_numpy(); view = g["View Position"].to_numpy()
        if lab.min() == lab.max():
            continue
        anchors = []
        for i in range(len(g)):
            other = np.arange(len(g)) != i
            fl = np.where(other & (lab != lab[i]))[0]
            sm = np.where(other & (lab == lab[i]))[0]
            if len(fl) == 0 or len(sm) == 0:
                continue
            f = fl[np.abs(fu[fl] - fu[i]).argmin()]
            s = sm[np.abs(fu[sm] - fu[i]).argmin()]
            anchors.append(dict(patient=pid, view=view[i], img_a=img[i], lab_a=int(lab[i]),
                                img_f=img[f], lab_f=int(lab[f]), img_s=img[s], lab_s=int(lab[s]),
                                gap_f=int(abs(fu[f] - fu[i])), gap_s=int(abs(fu[s] - fu[i]))))
        if anchors:
            idx = rng.permutation(len(anchors))[:max_anchors_per_patient]
            rows += [anchors[i] for i in idx]
    if same_view and rows:   # a patient can appear in two view groups: keep the per-patient cap
        t = pd.DataFrame(rows).sample(frac=1, random_state=seed)
        rows = t.groupby("patient").head(max_anchors_per_patient).to_dict("records")
    t = pd.DataFrame(rows)
    if len(t):
        t["direction"] = np.where(t["lab_a"] == 0, "absent->present", "present->absent")
    return t


def flip_counts(df, findings=NIH_FINDINGS, same_view=True):
    """Feasibility table: per finding, patients with a within-patient flip (each direction),
    and anchors with BOTH a flip and a same-label partner."""
    out = []
    key = ["Patient ID", "View Position"] if same_view else ["Patient ID"]
    for f in findings:
        d = add_label(df, f).sort_values(key + ["Follow-up #"])
        diff = d.groupby(key)["label"].diff()
        up = d.loc[diff == 1, "Patient ID"].nunique()      # consecutive studies, same view
        down = d.loc[diff == -1, "Patient ID"].nunique()
        trip = build_anchor_triples(df, f, same_view=same_view, max_anchors_per_patient=1)
        out.append(dict(finding=f, patients_absent_to_present=up, patients_present_to_absent=down,
                        patients_with_flip_and_same_partner=len(trip),
                        of_which_anchor_absent=int((trip["lab_a"] == 0).sum()) if len(trip) else 0,
                        of_which_anchor_present=int((trip["lab_a"] == 1).sum()) if len(trip) else 0))
    return pd.DataFrame(out)


def sample_balanced(trip, n, seed=0):
    """Balance the two flip directions so a yes/no bias cannot masquerade as reliance."""
    parts = [g.sample(min(len(g), n // 2), random_state=seed) for _, g in trip.groupby("direction")]
    return pd.concat(parts).reset_index(drop=True)


def item_quantities(res):
    """res needs p_a, p_f, p_s (P(yes) under anchor, flip partner, same-label partner)."""
    r = res.copy()
    r["d_change"] = (r["p_f"] - r["p_a"]).abs()          # sensitivity (should be large)
    r["d_preserve"] = (r["p_s"] - r["p_a"]).abs()        # nuisance sensitivity (should be small)
    r["R"] = r["d_change"] - r["d_preserve"]             # evidence-specific reliance
    r["signed_dir"] = (r["p_f"] - r["p_a"]) * (r["lab_f"] - r["lab_a"])  # >0 = moved the right way
    return r


def cluster_bootstrap(r, cols=("d_change", "d_preserve", "R", "signed_dir"), B=2000, seed=0, by="patient"):
    """Point estimates (mean of patient means) and percentile CIs resampling PATIENTS."""
    rng = np.random.default_rng(seed)
    pm = r.groupby(by)[list(cols)].mean()
    est = pm.mean()
    boots = np.empty((B, len(cols)))
    n = len(pm)
    arr = pm.to_numpy()
    for b in range(B):
        boots[b] = arr[rng.integers(0, n, n)].mean(axis=0)
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    return pd.DataFrame({"estimate": est.values, "ci_low": lo, "ci_high": hi, "n_patients": n}, index=list(cols))


def summarize(r, B=2000, seed=0):
    from scipy.stats import binomtest
    overall = cluster_bootstrap(r, B=B, seed=seed)
    by_dir = {k: cluster_bootstrap(g, B=B, seed=seed) for k, g in r.groupby("direction")}
    pm = r.groupby("patient")["R"].mean()
    nz = pm[pm != 0]
    sign_p = binomtest(int((nz > 0).sum()), len(nz), 0.5, alternative="greater").pvalue if len(nz) else np.nan
    acc = ((r["p_a"] > 0.5).astype(int) == r["lab_a"]).mean()
    return overall, by_dir, sign_p, acc


Writing pilot_lib.py


In [3]:
import os, glob, json, time
import numpy as np, pandas as pd
from pilot_lib import *

FINDING = "Effusion"            # NIH label used for the pilot
QUESTION_TERMS = {"Effusion": "a pleural effusion", "Pneumothorax": "a pneumothorax",
                  "Cardiomegaly": "cardiomegaly", "Atelectasis": "atelectasis",
                  "Consolidation": "consolidation", "Edema": "pulmonary edema",
                  "Pneumonia": "pneumonia", "Mass": "a lung mass", "Nodule": "a lung nodule"}
QUESTION = f"Is there {QUESTION_TERMS.get(FINDING, FINDING.lower())} in this chest X-ray? Answer yes or no."
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"   # ungated; swap for e.g. "google/medgemma-4b-it" (gated) later
N_ANCHORS = 300                 # balanced across the two flip directions
MAX_PIXELS = 512 * 28 * 28      # caps image tokens (~512) for speed on a T4
SEED = 0
OUT = "/kaggle/working"
print(QUESTION)

Is there a pleural effusion in this chest X-ray? Answer yes or no.


## 1. CheXTemporal: how many real 'new' / 'resolved' pairs exist?

In [4]:
try:
    gold = pd.read_parquet("hf://datasets/anonaccount107240/CheXTemporal/gold_progression_pairs.parquet")
    gold["progression"] = gold["progression"].astype(str).str.strip().str.capitalize()
    print(gold.shape, gold.columns.tolist())
    display(pd.crosstab(gold["disease_name"], gold["progression"], margins=True))
    display(pd.crosstab(gold["dataset"], gold["progression"], margins=True))
    flips = gold[gold["progression"].isin(["New", "Resolved"])]
    print("Gold flip pairs (New = absent->present, Resolved = present->absent):", len(flips),
          "| patients:", flips["patient_id"].nunique())
    display(pd.crosstab([flips["dataset"], flips["disease_name"]], flips["progression"]))
    gold.to_csv(f"{OUT}/chextemporal_gold_pairs.csv", index=False)
except Exception as e:
    print("Could not load CheXTemporal (is Internet on?):", repr(e))

(1787, 8) ['patient_id', 'study_id_prev', 'study_id_curr', 'img_path_prev', 'img_path_curr', 'disease_name', 'progression', 'dataset']


progression,Improved,New,Resolved,Stable,Worse,All
disease_name,,,,,,
atelectasis,57,12,12,60,59,200
cardiomegaly,13,4,1,99,16,133
consolidation,28,13,6,30,27,104
edema,67,8,5,85,57,222
enlarged cardiomediastinum,11,2,1,14,4,32
lung lesion,7,5,2,12,8,34
lung opacity,113,36,31,139,134,453
pleural effusion,95,54,37,168,101,455
pleural other,4,2,0,14,4,24


progression,Improved,New,Resolved,Stable,Worse,All
dataset,,,,,,
chexpert,250,105,68,391,260,1074
mimic,147,39,44,218,146,594
rexgradient,27,10,3,45,34,119
All,424,154,115,654,440,1787


Gold flip pairs (New = absent->present, Resolved = present->absent): 269 | patients: 90


progression                             New  Resolved
dataset     disease_name                             
chexpert    atelectasis                   6         4
            cardiomegaly                  2         0
            consolidation                11         5
            edema                         7         4
            lung lesion                   3         1
            lung opacity                 32        19
            pleural effusion             36        23
            pleural other                 1         0
            pneumonia                     2         1
            pneumothorax                  5        11
mimic       atelectasis                   6         8
            cardiomegaly                  2         1
            consolidation                 0         1
            edema                         1         1
            enlarged cardiomediastinum    1         1
            lung lesion                   1         0
            lung opacity                  3        11
            pleural effusion             16        14
            pleural other                 1         0
            pneumonia                     5         5
            pneumothorax                  3         2
rexgradient consolidation                 2         0
            enlarged cardiomediastinum    1         0
            lung lesion                   1         1
            lung opacity                  1         1
            pleural effusion              2         0
            pneumonia                     2         0
            pneumothorax                  1         1

In [5]:
# Silver set (automatic labels, ~280k rows): only for sizing a possible large-scale arm
try:
    silver = pd.read_parquet("hf://datasets/anonaccount107240/CheXTemporal/silver_findings.parquet")
    print(silver.shape, silver.columns.tolist())
    pcol = [c for c in silver.columns if "progress" in c.lower()]
    dcol = [c for c in silver.columns if c.lower() in ("dataset", "source")]
    with open(f"{OUT}/chextemporal_silver_columns.txt", "w") as fh:
        fh.write(" ".join(silver.columns))
    if pcol:
        tab = pd.crosstab(silver[dcol[0]] if dcol else silver[pcol[0]], silver[pcol[0]], margins=True)
        tab.to_csv(f"{OUT}/chextemporal_silver_counts.csv"); display(tab)
        fcol = [c for c in silver.columns if c.lower() in ("disease_name", "finding", "disease")]
        if fcol:
            t2 = pd.crosstab(silver[fcol[0]], [silver[dcol[0]], silver[pcol[0]]] if dcol else silver[pcol[0]])
            t2.to_csv(f"{OUT}/chextemporal_silver_counts_by_finding.csv")
except Exception as e:
    print("Silver set not loaded:", repr(e))

(282214, 16) ['dataset', 'patient_id', 'study_id_curr', 'study_id_prev', 'finding_token', 'silver_findings_path', 'silver_sentences_path', 'silver_mask_path', 'parent_image_curr', 'parent_image_prev', 'has_mask', 'finding', 'progression', 'anatomy', 'reasoning', 'evidence']


progression,Improved,New,Resolved,Stable,Worse,All
dataset,,,,,,
chexpert,22129,21536,1688,109912,42184,197449
mimic,13518,9969,1353,33120,21516,79476
rexgradient,953,767,87,2222,1260,5289
All,36600,32272,3128,145254,64960,282214


## 2. NIH ChestX-ray14: metadata and image paths

In [6]:
meta = [p for p in glob.glob("/kaggle/input/**/Data_Entry_2017*.csv", recursive=True)]
assert meta, "Add the NIH dataset (nih-chest-xrays/data) as an input"
df = pd.read_csv(meta[0])
df.columns = [c.strip() for c in df.columns]
print(meta[0], df.shape)
root = os.path.dirname(meta[0])
t = time.time()
img_paths = {os.path.basename(p): p for p in glob.glob(f"{root}/**/*.png", recursive=True)}
print(len(img_paths), "images indexed in", round(time.time() - t), "s")
print(df["View Position"].value_counts().to_dict())
print("patients:", df["Patient ID"].nunique(), "| with >= 3 studies:",
      (df.groupby("Patient ID").size() >= 3).sum())

/kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv (112120, 12)
112120 images indexed in 49 s
{'PA': 67310, 'AP': 44810}
patients: 30805 | with >= 3 studies: 9189


## 3. Feasibility table: within-patient flips per finding (same view)

In [7]:
t = time.time()
fc = flip_counts(df)
print(round(time.time() - t), "s")
fc.to_csv(f"{OUT}/nih_flip_counts.csv", index=False)
display(fc.sort_values("patients_with_flip_and_same_partner", ascending=False))

25 s


,finding,patients_absent_to_present,patients_present_to_absent,patients_with_flip_and_same_partner,of_which_anchor_absent,of_which_anchor_present
8,Infiltration,4623,4325,4625,3665,960
0,Atelectasis,3207,2971,3212,2547,665
4,Effusion,3059,2569,2993,2248,745
10,Nodule,1852,1904,1988,1654,334
2,Consolidation,1648,1547,1725,1550,175
9,Mass,1423,1477,1588,1256,332
11,Pleural_Thickening,1275,1222,1319,1181,138
13,Pneumothorax,1202,1142,1259,976,283
3,Edema,868,839,909,794,115
1,Cardiomegaly,794,815,839,744,95


## 4. Anchor triples for the pilot finding

In [8]:
trip = build_anchor_triples(df, FINDING, same_view=True, max_anchors_per_patient=2, seed=SEED)
print(len(trip), "triples |", trip["direction"].value_counts().to_dict())
samp = sample_balanced(trip, N_ANCHORS, seed=SEED)
samp = samp[samp[["img_a", "img_f", "img_s"]].isin(list(img_paths)).all(axis=1)].reset_index(drop=True)
print("sampled:", len(samp), "| patients:", samp["patient"].nunique(), "|", samp["direction"].value_counts().to_dict())
print("median follow-up gap  flip:", samp["gap_f"].median(), " same:", samp["gap_s"].median())
samp.to_csv(f"{OUT}/pilot_triples_{FINDING}.csv", index=False)

5986 triples | {'absent->present': 4517, 'present->absent': 1469}
sampled: 300 | patients: 290 | {'absent->present': 150, 'present->absent': 150}
median follow-up gap  flip: 2.0  same: 1.0


## 5. Model: P(yes) from next-token log-probabilities

In [9]:
import torch
from PIL import Image
from transformers import AutoProcessor
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as VLM
except ImportError:
    from transformers import AutoModelForImageTextToText as VLM

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

DTYPE = torch.float16           # if you see NaN below, set torch.float32
model = VLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto", token=token).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS, token=token)
tok = processor.tokenizer

def first_ids(words):
    return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES = first_ids(["Yes", "yes", " Yes", " yes", "YES"])
NO = first_ids(["No", "no", " No", " no", "NO"])
print("yes ids", YES, "no ids", NO)

@torch.no_grad()
def p_yes(path, question=QUESTION):
    img = Image.open(path).convert("RGB")
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[text], images=[img], return_tensors="pt").to(model.device)
    lp = torch.log_softmax(model(**inp).logits[0, -1].float(), -1)
    ly, ln = torch.logsumexp(lp[YES], 0), torch.logsumexp(lp[NO], 0)
    return torch.sigmoid(ly - ln).item(), (ly.exp() + ln.exp()).item()

# sanity check: P(yes) should be finite and most probability mass should sit on yes/no
for _, r in samp.head(3).iterrows():
    print(r["lab_a"], p_yes(img_paths[r["img_a"]]))

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

yes ids [7414, 9454, 9693, 9834, 14004] no ids [902, 2152, 2308, 2753, 8996]
0 (0.3488660156726837, 0.9986257553100586)
0 (0.35610267519950867, 0.9984185695648193)
0 (0.3596259653568268, 0.9985565543174744)


## 6. Score all images (cached per image, checkpointed)

In [10]:
cache_file = f"{OUT}/pyes_cache_{FINDING}_{MODEL_ID.split('/')[-1]}.json"
cache = json.load(open(cache_file)) if os.path.exists(cache_file) else {}
imgs = pd.unique(samp[["img_a", "img_f", "img_s"]].to_numpy().ravel())
t = time.time()
for i, im in enumerate(imgs):
    if im in cache:
        continue
    cache[im] = p_yes(img_paths[im])
    if i % 50 == 0:
        json.dump(cache, open(cache_file, "w"))
        print(f"{i}/{len(imgs)}  {time.time() - t:.0f}s")
json.dump(cache, open(cache_file, "w"))
print("done", len(cache), "images")

0/884  0s
50/884  25s
100/884  50s
150/884  76s
200/884  102s
250/884  130s
300/884  158s
350/884  185s
400/884  213s
450/884  240s
500/884  268s
550/884  295s
600/884  323s
650/884  350s
700/884  377s
750/884  405s
800/884  432s
850/884  459s
done 884 images


## 7. Estimates with patient-clustered bootstrap

In [11]:
res = samp.copy()
for c in ["a", "f", "s"]:
    res[f"p_{c}"] = res[f"img_{c}"].map(lambda k: cache[k][0])
    res[f"mass_{c}"] = res[f"img_{c}"].map(lambda k: cache[k][1])
print("mean probability mass on yes/no tokens:", round(res[["mass_a", "mass_f", "mass_s"]].mean().mean(), 3))

r = item_quantities(res)
overall, by_dir, sign_p, acc = summarize(r, B=2000, seed=SEED)
print(f"Anchor accuracy (threshold 0.5): {acc:.3f}")
print("\nOverall (S = d_change, N = d_preserve):"); display(overall.round(3))
for k, v in by_dir.items():
    print("\n", k); display(v.round(3))
print("One-sided sign test on patient-level R > 0: p =", sign_p)
r.to_csv(f"{OUT}/pilot_results_{FINDING}_{MODEL_ID.split('/')[-1]}.csv", index=False)

mean probability mass on yes/no tokens: 0.998
Anchor accuracy (threshold 0.5): 0.493

Overall (S = d_change, N = d_preserve):


,estimate,ci_low,ci_high,n_patients
d_change,0.051,0.047,0.056,290
d_preserve,0.051,0.046,0.056,290
R,0.000,-0.006,0.006,290
signed_dir,0.005,-0.003,0.012,290



 absent->present


,estimate,ci_low,ci_high,n_patients
d_change,0.051,0.044,0.059,150
d_preserve,0.050,0.043,0.056,150
R,0.002,-0.007,0.010,150
signed_dir,0.005,-0.006,0.016,150



 present->absent


,estimate,ci_low,ci_high,n_patients
d_change,0.052,0.046,0.058,145
d_preserve,0.052,0.045,0.060,145
R,-0.001,-0.010,0.008,145
signed_dir,0.004,-0.007,0.014,145


One-sided sign test on patient-level R > 0: p = 0.7013849618191781


## 8. How to read the output
- **R** with a CI above 0: the model's answer moves more when the finding changes than when an equally different-day image of the same patient keeps the finding status. That is evidence-specific reliance.
- **S high but R near 0**: the model reacts to any change of image, not to the finding (generic sensitivity).
- **direction > 0**: when it moves, it moves toward the right answer.
- Compare the two directions: reliance only for absent->present (or only present->absent) points to a yes/no bias.
- Check the **yes/no probability mass**; if it is low, the prompt is not being followed and P(yes) is unreliable.

**Send back:** `nih_flip_counts.csv`, the CheXTemporal tables, and the printed estimates.